# Preparing EuroSAT RGB

Each HDF5 row holds one 64 × 64 RGB tile, its filename, its original class ID and its assigned split. The split comes from `analysis/outputs/eurosat_split.csv` 

In [2]:
from pathlib import Path
import sys

import h5py
import numpy as np
import pandas as pd
import torch
from PIL import Image

root = Path.cwd()
if root.name == "analysis":
    root = root.parent
sys.path.insert(0, str(root))

from shared.prepare_data import prepare_data
from shared.dataset import make_loader
from station1.models import VIT, input_config

data = root / "data"
manifest = pd.read_csv(data / "eurosat_split.csv")
counts = pd.crosstab(manifest["class"], manifest["split"])[["train", "validation", "test"]]

assert np.allclose(counts.div(counts.sum(axis=1), axis=0), [0.8, 0.1, 0.1])

print(counts.to_string())

split                 train  validation  test
class                                        
AnnualCrop             2400         300   300
Forest                 2400         300   300
HerbaceousVegetation   2400         300   300
Highway                2000         250   250
Industrial             2000         250   250
Pasture                1600         200   200
PermanentCrop          2000         250   250
Residential            2400         300   300
River                  2000         250   250
SeaLake                2400         300   300


/Users/mansour/Library/Caches/pypoetry/virtualenvs/vitransformer-7ORiU7Kd-py3.15/lib/python3.15/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
path = prepare_data()

with h5py.File(path, "r") as h5:
    images = h5["images"]
    assert images.shape == (27000, 64, 64, 3) and images.dtype == np.uint8
    
    for key, column in [("filenames", "file"), ("split", "split")]:
        np.testing.assert_array_equal(h5[key].asstr()[:], manifest[column])
    
    np.testing.assert_array_equal(h5["labels"][:], manifest["label"])
    
    i = manifest.index[manifest["split"] == "train"][0]
    
    with Image.open(data / "EuroSAT_RGB" / manifest.at[i, "file"]) as source:
        np.testing.assert_array_equal(images[i], np.asarray(source))
    
    print(f"{len(images):,} original tiles in {path.name}")

27,000 original tiles in eurosat.h5


## Model inputs

A direct resize maps the entire 64 × 64 tile to 224 × 224. 

The checks below use deterministic validation inputs.

In [4]:
for model in ["resnet18", VIT]:
    loader = make_loader("validation", input_config(model), batch_size=8, path=path)
    images, labels = next(iter(loader))
    
    assert images.shape == (8, 3, 224, 224) and images.dtype == torch.float32
    assert torch.isfinite(images).all()
    
    torch.testing.assert_close(images[0], loader.dataset[0][0])
    print(model, tuple(images.shape), labels.tolist())

resnet18 (8, 3, 224, 224) [0, 0, 0, 0, 0, 0, 0, 0]
vit_small_patch16_224.augreg_in21k_ft_in1k (8, 3, 224, 224) [0, 0, 0, 0, 0, 0, 0, 0]
